# 🎒 Bagging — Bootstrap Aggregating

> **Folder:** `08_Ensemble_Learning`  
> **Notebook:** `bagging.ipynb`  
> **Author:** Hamna Munir

---

## 🎯 Objectives

By the end of this notebook, you will:

- Understand **how bagging reduces variance** without increasing bias
- Implement **BaggingClassifier and BaggingRegressor** from scratch
- Use **Out-of-Bag (OOB) evaluation** as a free internal validation
- Compare **bagging vs single model** on bias-variance tradeoff
- Tune bagging hyperparameters: `n_estimators`, `max_samples`, `max_features`
- Understand **Pasting vs Bagging** (with vs without replacement)
- Build **Random Subspaces** and **Random Patches** ensembles
- Visualize **diversity** among base estimators

---

## 📚 Techniques Covered

| # | Technique | Key Insight |
|---|-----------|-------------|
| 1 | Dataset Setup | Classification + Regression |
| 2 | Bagging Intuition | Variance reduction via averaging |
| 3 | BaggingClassifier | Core usage + OOB score |
| 4 | BaggingRegressor | Regression bagging + residual analysis |
| 5 | n_estimators Sensitivity | How many bags are enough? |
| 6 | max_samples Sensitivity | Bootstrap sample size effect |
| 7 | Pasting vs Bagging | With vs without replacement |
| 8 | Random Subspaces | Feature sampling per estimator |
| 9 | Random Patches | Both sample + feature subsampling |
| 10 | OOB Evaluation | Free internal validation |
| 11 | Bias-Variance Decomposition | Why bagging helps high-variance models |
| 12 | Summary & Golden Rules | Key takeaways |


---
## ⚙️ 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, KFold,
    cross_val_score,
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import (
    BaggingClassifier, BaggingRegressor,
    RandomForestClassifier,
)
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    r2_score, mean_squared_error,
)

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)

COLORS = {
    'primary'  : '#2E86AB',
    'secondary': '#E84855',
    'accent'   : '#3BB273',
    'warning'  : '#F18F01',
    'purple'   : '#7B2D8B',
    'palette'  : ['#2E86AB','#E84855','#3BB273','#F18F01','#7B2D8B','#F4D35E'],
}
print('✅ Libraries loaded successfully!')

---
## 1️⃣ Dataset Setup


In [ ]:
np.random.seed(42)

# ── Binary classification ─────────────────────────────────────────────────
X_clf, y_clf = make_classification(
    n_samples=1000, n_features=20, n_informative=10,
    n_redundant=5, n_classes=2, weights=[0.5, 0.5],
    random_state=42
)
X_clf_df = pd.DataFrame(X_clf, columns=[f'F{i+1:02d}' for i in range(20)])
y_clf_s  = pd.Series(y_clf, name='Target')

# ── Regression ────────────────────────────────────────────────────────────
X_reg, y_reg = make_regression(
    n_samples=800, n_features=15, n_informative=8,
    noise=25, random_state=42
)
X_reg_df = pd.DataFrame(X_reg, columns=[f'R{i+1:02d}' for i in range(15)])
y_reg_s  = pd.Series(y_reg, name='Target')

# ── Scale + split ─────────────────────────────────────────────────────────
sc_clf = StandardScaler(); sc_reg = StandardScaler()

Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(
    X_clf_df, y_clf_s, test_size=0.2, stratify=y_clf_s, random_state=42)
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(
    X_reg_df, y_reg_s, test_size=0.2, random_state=42)

Xc_tr_sc = sc_clf.fit_transform(Xc_tr); Xc_te_sc = sc_clf.transform(Xc_te)
Xr_tr_sc = sc_reg.fit_transform(Xr_tr); Xr_te_sc = sc_reg.transform(Xr_te)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
kf  = KFold(n_splits=5, shuffle=True, random_state=42)

print(f'Classification: {X_clf_df.shape} | classes={dict(y_clf_s.value_counts().sort_index())}')
print(f'  Train={Xc_tr.shape}  Test={Xc_te.shape}')
print(f'Regression    : {X_reg_df.shape}')
print(f'  Train={Xr_tr.shape}  Test={Xr_te.shape}')

---
## 2️⃣ Bagging Intuition — Variance Reduction via Averaging

> **Bagging = Bootstrap AGGregating**
>
> ```
> Given: Training set D of n samples
>
> For each estimator b = 1, 2, ..., B:
>   1. Draw bootstrap sample Dᵦ of n samples WITH replacement from D
>      (~63.2% unique samples, ~36.8% duplicates)
>   2. Train base estimator hᵦ on Dᵦ
>
> Final prediction:
>   Classification: majority vote  ŷ = mode(h₁(x), h₂(x), ..., hB(x))
>   Regression:     average        ŷ = (1/B) Σ hᵦ(x)
> ```
>
> **Why it works:** Each estimator sees a different bootstrap sample →  
> diverse estimators → averaging cancels out individual errors →  
> **Variance ↓, Bias unchanged**


In [ ]:
# Demonstrate bootstrap sampling
n_samples   = 200
n_bootstrap = 5

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Show bootstrap sample overlap
all_indices  = np.arange(n_samples)
unique_fracs = []
for b in range(200):
    boot_idx = np.random.choice(n_samples, size=n_samples, replace=True)
    unique_fracs.append(len(set(boot_idx)) / n_samples)

axes[0].hist(unique_fracs, bins=30, color=COLORS['primary'],
             alpha=0.80, edgecolor='white')
axes[0].axvline(np.mean(unique_fracs), color=COLORS['secondary'],
                linestyle='--', linewidth=2.5,
                label=f'Mean unique = {np.mean(unique_fracs)*100:.1f}%')
axes[0].axvline(1 - 1/np.e, color=COLORS['accent'],
                linestyle=':', linewidth=2,
                label=f'Theoretical = {(1-1/np.e)*100:.1f}%')
axes[0].set_xlabel('Fraction of Unique Samples', fontsize=11)
axes[0].set_ylabel('Count (over 200 bootstrap draws)', fontsize=11)
axes[0].set_title('Bootstrap Sample Uniqueness
~63.2% unique per bag',
                  fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)

# Show variance reduction with averaging
n_estimators_range = range(1, 101)
single_var = 0.25  # variance of a single noisy estimator
ensemble_var = [single_var / b for b in n_estimators_range]
axes[1].plot(n_estimators_range, ensemble_var, '-',
             color=COLORS['primary'], linewidth=2.5)
axes[1].axhline(single_var, color=COLORS['secondary'], linestyle='--',
                linewidth=2, label=f'Single model variance={single_var}')
for b in [5, 10, 20, 50]:
    v = single_var / b
    axes[1].scatter(b, v, s=100, zorder=5, color=COLORS['accent'])
    axes[1].annotate(f'B={b}
var={v:.3f}',
                     (b, v), textcoords='offset points',
                     xytext=(8, 5), fontsize=8)
axes[1].set_xlabel('Number of Estimators (B)', fontsize=11)
axes[1].set_ylabel('Ensemble Variance (σ²/B)', fontsize=11)
axes[1].set_title('Variance Reduction via Averaging
(assumes independent estimators)',
                  fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)

plt.suptitle('Bagging — Bootstrap Sampling and Variance Reduction',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Theoretical unique fraction per bootstrap: {(1-1/np.e)*100:.2f}%')
print(f'Observed mean unique fraction: {np.mean(unique_fracs)*100:.2f}%')

---
## 3️⃣ BaggingClassifier — Core Usage

> `BaggingClassifier` wraps any base estimator and trains B bootstrap copies.
>
> Key parameters:
> - `estimator` — base learner (default: DecisionTreeClassifier)
> - `n_estimators` — number of bags (more = better, diminishing returns)
> - `max_samples` — fraction of train set per bag (default: 1.0)
> - `max_features` — fraction of features per bag (default: 1.0)
> - `bootstrap` — with replacement (True=Bagging, False=Pasting)
> - `oob_score` — use out-of-bag samples for free validation


In [ ]:
# ── Single deep tree (high variance) ─────────────────────────────────────
single_tree = DecisionTreeClassifier(max_depth=None, random_state=42)
single_tree.fit(Xc_tr_sc, yc_tr)
single_auc  = roc_auc_score(yc_te, single_tree.predict_proba(Xc_te_sc)[:,1])
single_acc  = accuracy_score(yc_te, single_tree.predict(Xc_te_sc))
single_tr_auc = roc_auc_score(yc_tr, single_tree.predict_proba(Xc_tr_sc)[:,1])

print('Single Deep Decision Tree (no bagging):')
print(f'  Train AUC: {single_tr_auc:.4f}')
print(f'  Test AUC : {single_auc:.4f}')
print(f'  Test Acc : {single_acc:.4f}')
print(f'  Overfit  : {single_tr_auc - single_auc:.4f}')
print()

# ── BaggingClassifier ─────────────────────────────────────────────────────
bag_clf = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=None),
    n_estimators=100,
    max_samples=1.0,       # use full training set per bag (with replacement)
    max_features=1.0,      # use all features per bag
    bootstrap=True,        # sample with replacement (True = Bagging)
    bootstrap_features=False,
    oob_score=True,        # free OOB validation
    n_jobs=-1,
    random_state=42,
)
bag_clf.fit(Xc_tr_sc, yc_tr)

bag_auc    = roc_auc_score(yc_te, bag_clf.predict_proba(Xc_te_sc)[:,1])
bag_acc    = accuracy_score(yc_te, bag_clf.predict(Xc_te_sc))
bag_tr_auc = roc_auc_score(yc_tr, bag_clf.predict_proba(Xc_tr_sc)[:,1])

print('BaggingClassifier (100 trees, bootstrap=True):')
print(f'  Train AUC : {bag_tr_auc:.4f}')
print(f'  Test AUC  : {bag_auc:.4f}')
print(f'  Test Acc  : {bag_acc:.4f}')
print(f'  Overfit   : {bag_tr_auc - bag_auc:.4f}')
print(f'  OOB Score : {bag_clf.oob_score_:.4f}  ← free validation (no test set needed!)')
print(f'  AUC gain  : {bag_auc - single_auc:+.4f} over single tree')

# Comparison bar
fig, ax = plt.subplots(figsize=(10, 5))
models  = ['Single Deep Tree', 'BaggingClassifier
(100 trees)']
tr_aucs = [single_tr_auc, bag_tr_auc]
te_aucs = [single_auc,    bag_auc]
x = np.arange(2); w = 0.35
ax.bar(x - w/2, tr_aucs, w, label='Train AUC',
       color=COLORS['primary'], alpha=0.85)
ax.bar(x + w/2, te_aucs, w, label='Test AUC',
       color=COLORS['accent'], alpha=0.85)
for i, (tr, te) in enumerate(zip(tr_aucs, te_aucs)):
    ax.text(i-w/2, tr+0.003, f'{tr:.4f}', ha='center', fontsize=11)
    ax.text(i+w/2, te+0.003, f'{te:.4f}', ha='center', fontsize=11)
ax.set_xticks(x); ax.set_xticklabels(models, fontsize=11)
ax.set_ylabel('ROC-AUC', fontsize=11)
ax.set_title('Single Tree vs BaggingClassifier
Variance Reduction in Action',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.set_ylim([0.7, 1.05])
plt.tight_layout(); plt.show()

---
## 4️⃣ BaggingRegressor — Regression Bagging

> Bagging averages predictions across B bootstrap regressors,  
> reducing variance without adding bias.


In [ ]:
# ── Single deep regression tree ───────────────────────────────────────────
single_rtr = DecisionTreeRegressor(max_depth=None, random_state=42)
single_rtr.fit(Xr_tr_sc, yr_tr)
r2_single  = r2_score(yr_te, single_rtr.predict(Xr_te_sc))
rmse_single = np.sqrt(mean_squared_error(yr_te, single_rtr.predict(Xr_te_sc)))

# ── BaggingRegressor ──────────────────────────────────────────────────────
bag_reg = BaggingRegressor(
    estimator=DecisionTreeRegressor(max_depth=None),
    n_estimators=100,
    max_samples=1.0,
    bootstrap=True,
    oob_score=True,
    n_jobs=-1,
    random_state=42,
)
bag_reg.fit(Xr_tr_sc, yr_tr)
r2_bag   = r2_score(yr_te, bag_reg.predict(Xr_te_sc))
rmse_bag = np.sqrt(mean_squared_error(yr_te, bag_reg.predict(Xr_te_sc)))

print('Regression Comparison:')
print(f'  Single Tree  : R²={r2_single:.4f} | RMSE={rmse_single:.4f}')
print(f'  BaggingReg   : R²={r2_bag:.4f}   | RMSE={rmse_bag:.4f}')
print(f'  OOB R²       : {bag_reg.oob_score_:.4f}')
print(f'  R² gain      : {r2_bag - r2_single:+.4f}')

# Residual plots
y_pred_single = single_rtr.predict(Xr_te_sc)
y_pred_bag    = bag_reg.predict(Xr_te_sc)

fig, axes = plt.subplots(1, 3, figsize=(19, 5))

# Actual vs Predicted
for ax, (name, y_pred, r2) in zip(axes[:2], [
    ('Single Tree',     y_pred_single, r2_single),
    ('BaggingRegressor', y_pred_bag,   r2_bag),
]):
    ax.scatter(yr_te, y_pred, color=COLORS['primary'],
               alpha=0.45, s=25, edgecolors='none')
    mn = min(yr_te.min(), y_pred.min())
    mx = max(yr_te.max(), y_pred.max())
    ax.plot([mn, mx], [mn, mx], 'r--', linewidth=2, label='Perfect prediction')
    ax.set_xlabel('Actual', fontsize=11)
    ax.set_ylabel('Predicted', fontsize=11)
    ax.set_title(f'{name}
R²={r2:.4f}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)

# Residuals comparison
resid_single = yr_te.values - y_pred_single
resid_bag    = yr_te.values - y_pred_bag
axes[2].hist(resid_single, bins=30, alpha=0.60, color=COLORS['secondary'],
             edgecolor='white', label=f'Single Tree (std={resid_single.std():.1f})')
axes[2].hist(resid_bag, bins=30, alpha=0.60, color=COLORS['accent'],
             edgecolor='white', label=f'Bagging (std={resid_bag.std():.1f})')
axes[2].axvline(0, color='black', linewidth=1.5, linestyle='--')
axes[2].set_xlabel('Residual', fontsize=11)
axes[2].set_ylabel('Count', fontsize=11)
axes[2].set_title('Residual Distribution
Single Tree vs Bagging',
                  fontsize=12, fontweight='bold')
axes[2].legend(fontsize=10)

plt.suptitle('BaggingRegressor — Variance Reduction', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 5️⃣ n_estimators Sensitivity — How Many Bags Are Enough?

> More estimators = lower variance, but with diminishing returns.  
> The error curve flattens — adding bags past a certain point wastes compute.


In [ ]:
n_est_values = [1, 2, 5, 10, 20, 30, 50, 75, 100, 150, 200]
clf_scores, reg_scores = [], []
oob_clf, oob_reg = [], []

for n in n_est_values:
    # Classification
    bc = BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=None),
        n_estimators=n, bootstrap=True,
        oob_score=(n > 1), n_jobs=-1, random_state=42
    )
    bc.fit(Xc_tr_sc, yc_tr)
    clf_scores.append(roc_auc_score(yc_te, bc.predict_proba(Xc_te_sc)[:,1]))
    oob_clf.append(bc.oob_score_ if n > 1 else np.nan)

    # Regression
    br = BaggingRegressor(
        estimator=DecisionTreeRegressor(max_depth=None),
        n_estimators=n, bootstrap=True,
        oob_score=(n > 1), n_jobs=-1, random_state=42
    )
    br.fit(Xr_tr_sc, yr_tr)
    reg_scores.append(r2_score(yr_te, br.predict(Xr_te_sc)))
    oob_reg.append(br.oob_score_ if n > 1 else np.nan)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(n_est_values, clf_scores, 'o-',
             color=COLORS['primary'], linewidth=2.5, markersize=7,
             label='Test ROC-AUC')
axes[0].plot(n_est_values, oob_clf, 's--',
             color=COLORS['secondary'], linewidth=2, markersize=6,
             label='OOB Score')
axes[0].axhline(single_auc, color=COLORS['warning'], linestyle=':',
                linewidth=2, label=f'Single tree ({single_auc:.3f})')
axes[0].set_xlabel('n_estimators', fontsize=11)
axes[0].set_ylabel('ROC-AUC', fontsize=11)
axes[0].set_title('Classification — n_estimators Sensitivity',
                  fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)

axes[1].plot(n_est_values, reg_scores, 'o-',
             color=COLORS['accent'], linewidth=2.5, markersize=7,
             label='Test R²')
axes[1].plot(n_est_values, oob_reg, 's--',
             color=COLORS['secondary'], linewidth=2, markersize=6,
             label='OOB R²')
axes[1].axhline(r2_single, color=COLORS['warning'], linestyle=':',
                linewidth=2, label=f'Single tree ({r2_single:.3f})')
axes[1].set_xlabel('n_estimators', fontsize=11)
axes[1].set_ylabel('R²', fontsize=11)
axes[1].set_title('Regression — n_estimators Sensitivity',
                  fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)

plt.suptitle('Bagging — n_estimators Sensitivity', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

# Find elbow (point of diminishing returns)
gains = np.diff(clf_scores)
elbow = n_est_values[np.argmin(np.abs(gains)) + 1]
print(f'Classification elbow (approx): n_estimators ≈ {elbow}')
print(f'Best test AUC at n=200: {max(clf_scores):.4f}')

---
## 6️⃣ max_samples — Bootstrap Sample Size Effect

> `max_samples` controls what fraction of the training set each bag uses.
>
> - `max_samples=1.0` → each bag sees n samples (default, full bootstrap)
> - `max_samples=0.5` → each bag sees n/2 samples → more diversity, less data per bag
>
> Smaller `max_samples` → more diverse estimators → more variance reduction  
> but each estimator trains on less data → higher individual bias


In [ ]:
max_samples_values = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
ms_results = []

for ms in max_samples_values:
    bc = BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=None),
        n_estimators=100, max_samples=ms,
        bootstrap=True, oob_score=True,
        n_jobs=-1, random_state=42
    )
    bc.fit(Xc_tr_sc, yc_tr)
    tr_auc = roc_auc_score(yc_tr, bc.predict_proba(Xc_tr_sc)[:,1])
    te_auc = roc_auc_score(yc_te, bc.predict_proba(Xc_te_sc)[:,1])
    ms_results.append({
        'max_samples': ms,
        'Train AUC'  : round(tr_auc, 4),
        'Test AUC'   : round(te_auc, 4),
        'OOB Score'  : round(bc.oob_score_, 4),
        'Overfit Gap': round(tr_auc - te_auc, 4),
    })

ms_df = pd.DataFrame(ms_results)
print('max_samples Sensitivity (n_estimators=100):')
print(ms_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].plot(ms_df['max_samples'], ms_df['Train AUC'], 'o-',
             color=COLORS['primary'], linewidth=2.5, markersize=8,
             label='Train AUC')
axes[0].plot(ms_df['max_samples'], ms_df['Test AUC'], 's-',
             color=COLORS['accent'], linewidth=2.5, markersize=8,
             label='Test AUC')
axes[0].plot(ms_df['max_samples'], ms_df['OOB Score'], '^--',
             color=COLORS['secondary'], linewidth=2, markersize=7,
             label='OOB Score')
axes[0].set_xlabel('max_samples', fontsize=11)
axes[0].set_ylabel('ROC-AUC', fontsize=11)
axes[0].set_title('max_samples — Train vs Test vs OOB',
                  fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)

axes[1].bar(ms_df['max_samples'].astype(str), ms_df['Overfit Gap'],
            color=[COLORS['secondary'] if g > 0.05 else COLORS['accent']
                   for g in ms_df['Overfit Gap']],
            alpha=0.85, edgecolor='white')
axes[1].axhline(0.05, color=COLORS['warning'], linestyle='--',
                linewidth=2, label='Overfit threshold (0.05)')
axes[1].set_xlabel('max_samples', fontsize=11)
axes[1].set_ylabel('Train − Test AUC', fontsize=11)
axes[1].set_title('Overfitting Gap vs max_samples', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)

plt.suptitle('Bagging — max_samples Sensitivity', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 7️⃣ Pasting vs Bagging — With vs Without Replacement

> - **Bagging** (`bootstrap=True`): sample WITH replacement → duplicates allowed
> - **Pasting** (`bootstrap=False`): sample WITHOUT replacement → unique subsets
>
> Bagging is generally preferred — more randomness → more diverse estimators.  
> Pasting works well when training data is very large (each subset still varied).


In [ ]:
results_boot = []
for n in [10, 25, 50, 100, 150]:
    for boot, label in [(True, 'Bagging'), (False, 'Pasting')]:
        bc = BaggingClassifier(
            estimator=DecisionTreeClassifier(max_depth=None),
            n_estimators=n, max_samples=0.7,
            bootstrap=boot, n_jobs=-1, random_state=42
        )
        bc.fit(Xc_tr_sc, yc_tr)
        auc = roc_auc_score(yc_te, bc.predict_proba(Xc_te_sc)[:,1])
        results_boot.append({
            'Method'      : label,
            'n_estimators': n,
            'Test AUC'    : round(auc, 4),
        })

boot_df = pd.DataFrame(results_boot)
bagging_df = boot_df[boot_df['Method']=='Bagging']
pasting_df = boot_df[boot_df['Method']=='Pasting']

print('Bagging vs Pasting (max_samples=0.7):')
print(boot_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(bagging_df['n_estimators'], bagging_df['Test AUC'], 'o-',
        color=COLORS['primary'], linewidth=2.5, markersize=9,
        label='Bagging (bootstrap=True)')
ax.plot(pasting_df['n_estimators'], pasting_df['Test AUC'], 's--',
        color=COLORS['secondary'], linewidth=2.5, markersize=9,
        label='Pasting (bootstrap=False)')
ax.set_xlabel('n_estimators', fontsize=11)
ax.set_ylabel('Test ROC-AUC', fontsize=11)
ax.set_title('Bagging vs Pasting — Test AUC vs n_estimators
(max_samples=0.7)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout(); plt.show()

---
## 8️⃣ Random Subspaces — Feature Sampling per Estimator

> **Random Subspaces**: sample features (not samples) per estimator.
> - `bootstrap=False`, `bootstrap_features=True`
> - Each estimator sees a different random subset of features
> - Increases diversity → reduces correlation between estimators
>
> This is related to how Random Forest works internally.


In [ ]:
max_features_values = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
subspace_results = []

for mf in max_features_values:
    # Random Subspaces (feature sampling, no sample sampling)
    bc_rs = BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=None),
        n_estimators=100,
        max_samples=1.0,          # use all samples
        max_features=mf,          # random feature subset
        bootstrap=False,          # no sample replacement
        bootstrap_features=True,  # feature sampling with replacement
        n_jobs=-1, random_state=42
    )
    bc_rs.fit(Xc_tr_sc, yc_tr)
    auc_rs = roc_auc_score(yc_te, bc_rs.predict_proba(Xc_te_sc)[:,1])

    # Standard Bagging for comparison
    bc_std = BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=None),
        n_estimators=100, max_samples=1.0,
        max_features=mf, bootstrap=True,
        bootstrap_features=False,
        n_jobs=-1, random_state=42
    )
    bc_std.fit(Xc_tr_sc, yc_tr)
    auc_std = roc_auc_score(yc_te, bc_std.predict_proba(Xc_te_sc)[:,1])

    subspace_results.append({
        'max_features'  : mf,
        'Random Subspace': round(auc_rs, 4),
        'Standard Bagging': round(auc_std, 4),
    })

sub_df = pd.DataFrame(subspace_results)
print('Random Subspaces vs Standard Bagging:')
print(sub_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(sub_df['max_features'], sub_df['Random Subspace'], 'o-',
        color=COLORS['purple'], linewidth=2.5, markersize=9,
        label='Random Subspaces (feature sampling)')
ax.plot(sub_df['max_features'], sub_df['Standard Bagging'], 's--',
        color=COLORS['primary'], linewidth=2.5, markersize=9,
        label='Standard Bagging (sample sampling)')
ax.set_xlabel('max_features fraction', fontsize=11)
ax.set_ylabel('Test ROC-AUC', fontsize=11)
ax.set_title('Random Subspaces vs Standard Bagging
vs max_features',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout(); plt.show()

---
## 9️⃣ Random Patches — Sample AND Feature Subsampling

> **Random Patches** combines both:
> - Bootstrap sample subsets (`bootstrap=True`, `max_samples < 1.0`)
> - Feature subsets (`bootstrap_features=True`, `max_features < 1.0`)
>
> Maximum diversity → lowest correlation between estimators →  
> strongest variance reduction, especially on high-dimensional data.


In [ ]:
configs = [
    ('Standard Bagging',   True,  False, 1.0, 1.0),
    ('Random Subspaces',   False, True,  1.0, 0.5),
    ('Pasting',            False, False, 0.7, 1.0),
    ('Random Patches',     True,  True,  0.7, 0.5),
    ('Random Forest (ref)',None,  None,  None, None),
]

patch_results = []
for name, boot, boot_feat, ms, mf in configs:
    if name == 'Random Forest (ref)':
        model = RandomForestClassifier(
            n_estimators=100, random_state=42, n_jobs=-1)
        model.fit(Xc_tr_sc, yc_tr)
    else:
        model = BaggingClassifier(
            estimator=DecisionTreeClassifier(max_depth=None),
            n_estimators=100,
            max_samples=ms, max_features=mf,
            bootstrap=boot, bootstrap_features=boot_feat,
            n_jobs=-1, random_state=42
        )
        model.fit(Xc_tr_sc, yc_tr)

    tr_auc = roc_auc_score(yc_tr, model.predict_proba(Xc_tr_sc)[:,1])
    te_auc = roc_auc_score(yc_te, model.predict_proba(Xc_te_sc)[:,1])
    patch_results.append({
        'Method'    : name,
        'Train AUC' : round(tr_auc, 4),
        'Test AUC'  : round(te_auc, 4),
        'Overfit'   : round(tr_auc - te_auc, 4),
    })
    print(f'  {name:30s}: Train={tr_auc:.4f} | Test={te_auc:.4f} '
          f'| Gap={tr_auc-te_auc:.4f}')

patch_df = pd.DataFrame(patch_results).sort_values('Test AUC', ascending=False)
print('
Summary (sorted by Test AUC):')
print(patch_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(patch_df)); w = 0.35
ax.bar(x - w/2, patch_df['Train AUC'], w, label='Train AUC',
       color=COLORS['primary'], alpha=0.85)
ax.bar(x + w/2, patch_df['Test AUC'],  w, label='Test AUC',
       color=COLORS['accent'], alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(patch_df['Method'], rotation=20, ha='right', fontsize=9)
ax.set_ylabel('ROC-AUC', fontsize=11)
ax.set_title('Bagging Variants — Train vs Test AUC Comparison',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.set_ylim([0.7, 1.05])
plt.tight_layout(); plt.show()

---
## 🔟 Out-of-Bag (OOB) Evaluation — Free Internal Validation

> Each bootstrap sample leaves out ~36.8% of training points.  
> These **out-of-bag** samples can be used to evaluate the model —  
> without needing a separate validation set!
>
> OOB score ≈ leave-one-out CV score — very reliable estimate.


In [ ]:
# Compare OOB score vs CV score vs Test score
n_est_list   = [10, 20, 50, 100, 200]
oob_scores   = []
cv_scores    = []
test_scores  = []

for n in n_est_list:
    bc = BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=None),
        n_estimators=n, bootstrap=True,
        oob_score=True, n_jobs=-1, random_state=42
    )
    bc.fit(Xc_tr_sc, yc_tr)
    oob_scores.append(bc.oob_score_)
    test_scores.append(accuracy_score(yc_te, bc.predict(Xc_te_sc)))

    cv_sc = cross_val_score(
        BaggingClassifier(
            estimator=DecisionTreeClassifier(max_depth=None),
            n_estimators=n, bootstrap=True,
            n_jobs=-1, random_state=42
        ),
        Xc_tr_sc, yc_tr, cv=skf, scoring='accuracy'
    )
    cv_scores.append(cv_sc.mean())

oob_df = pd.DataFrame({
    'n_estimators': n_est_list,
    'OOB Score'   : [round(s,4) for s in oob_scores],
    'CV Score'    : [round(s,4) for s in cv_scores],
    'Test Score'  : [round(s,4) for s in test_scores],
})
print('OOB vs CV vs Test Accuracy:')
print(oob_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(n_est_list, oob_scores, 'o-', color=COLORS['primary'],
        linewidth=2.5, markersize=9, label='OOB Score (free!)')
ax.plot(n_est_list, cv_scores, 's--', color=COLORS['secondary'],
        linewidth=2.5, markersize=9, label='5-Fold CV Score')
ax.plot(n_est_list, test_scores, '^:', color=COLORS['accent'],
        linewidth=2.5, markersize=9, label='Test Score')
ax.set_xlabel('n_estimators', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title('OOB Score vs CV vs Test Score
(OOB is a free, reliable estimate)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout(); plt.show()

# OOB decision function per sample
bc_oob = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=None),
    n_estimators=200, bootstrap=True,
    oob_score=True, n_jobs=-1, random_state=42
)
bc_oob.fit(Xc_tr_sc, yc_tr)
oob_probs = bc_oob.oob_decision_function_[:, 1]
print(f'
OOB decision function shape : {bc_oob.oob_decision_function_.shape}')
print(f'OOB AUC (training data only): '
      f'{roc_auc_score(yc_tr, oob_probs):.4f}')
print(f'Test AUC                    : '
      f'{roc_auc_score(yc_te, bc_oob.predict_proba(Xc_te_sc)[:,1]):.4f}')

---
## 1️⃣1️⃣ Bias-Variance — Why Bagging Helps High-Variance Models

> Bagging is most effective when the base estimator has **high variance**  
> (overfits the training data). It reduces variance without increasing bias.
>
> Low-variance models (e.g. LogisticRegression) benefit little from bagging.


In [ ]:
base_estimators = {
    'Decision Tree (deep)' : DecisionTreeClassifier(max_depth=None),
    'Decision Tree (d=3)'  : DecisionTreeClassifier(max_depth=3),
    'KNN (k=1)'            : KNeighborsClassifier(n_neighbors=1),
    'KNN (k=5)'            : KNeighborsClassifier(n_neighbors=5),
    'LogisticRegression'   : LogisticRegression(max_iter=1000),
    'SVM (RBF)'            : SVC(probability=True),
}

bv_rows = []
for name, estimator in base_estimators.items():
    # Single model
    estimator.fit(Xc_tr_sc, yc_tr)
    tr_single = roc_auc_score(yc_tr, estimator.predict_proba(Xc_tr_sc)[:,1])
    te_single = roc_auc_score(yc_te, estimator.predict_proba(Xc_te_sc)[:,1])

    # Bagged model
    bag = BaggingClassifier(
        estimator=estimator.__class__(**estimator.get_params()),
        n_estimators=50, bootstrap=True,
        n_jobs=-1, random_state=42
    )
    bag.fit(Xc_tr_sc, yc_tr)
    tr_bag = roc_auc_score(yc_tr, bag.predict_proba(Xc_tr_sc)[:,1])
    te_bag = roc_auc_score(yc_te, bag.predict_proba(Xc_te_sc)[:,1])

    bv_rows.append({
        'Estimator'     : name,
        'Single Test'   : round(te_single, 4),
        'Bagged Test'   : round(te_bag, 4),
        'Gain'          : round(te_bag - te_single, 4),
        'Single Overfit': round(tr_single - te_single, 4),
        'Bagged Overfit': round(tr_bag - te_bag, 4),
    })
    print(f'  {name:25s} | Single={te_single:.4f} | '
          f'Bagged={te_bag:.4f} | Gain={te_bag-te_single:+.4f}')

bv_df = pd.DataFrame(bv_rows)

fig, axes = plt.subplots(1, 2, figsize=(17, 5))
x = np.arange(len(bv_df)); w = 0.35
axes[0].bar(x-w/2, bv_df['Single Test'], w, label='Single Model',
            color=COLORS['secondary'], alpha=0.85)
axes[0].bar(x+w/2, bv_df['Bagged Test'], w, label='Bagged (50 estimators)',
            color=COLORS['accent'], alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(bv_df['Estimator'], rotation=25, ha='right', fontsize=9)
axes[0].set_ylabel('Test ROC-AUC', fontsize=11)
axes[0].set_title('Single vs Bagged — Test AUC by Estimator Type',
                  fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10); axes[0].set_ylim([0.5, 1.05])

gain_colors = [COLORS['accent'] if g > 0.01 else
               COLORS['warning'] if g > 0 else COLORS['secondary']
               for g in bv_df['Gain']]
axes[1].barh(bv_df['Estimator'], bv_df['Gain'],
             color=gain_colors, alpha=0.85, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=1)
axes[1].axvline(0.01, color='gray', linestyle='--', linewidth=1.5,
                label='Meaningful gain threshold')
axes[1].set_xlabel('AUC Gain from Bagging', fontsize=11)
axes[1].set_title('Gain from Bagging per Estimator
(high-variance models benefit most)',
                  fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)

plt.suptitle('Bias-Variance — Which Models Benefit Most from Bagging?',
             fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## ✅ 12. Summary & Golden Rules

| Variant | bootstrap | bootstrap_features | max_samples | max_features |
|---------|:---------:|:-----------------:|:-----------:|:------------:|
| **Bagging** | ✅ True | ❌ False | 1.0 | 1.0 |
| **Pasting** | ❌ False | ❌ False | < 1.0 | 1.0 |
| **Random Subspaces** | ❌ False | ✅ True | 1.0 | < 1.0 |
| **Random Patches** | ✅ True | ✅ True | < 1.0 | < 1.0 |
| **Random Forest** | ✅ True | ✅ True | 1.0 | sqrt(d) |

### 🔑 Golden Rules

1. **Bagging reduces variance — not bias** — only helps high-variance (overfit) models
2. **Always enable `oob_score=True`** — free internal validation, no extra data needed
3. **n_estimators=100** is a solid default — more rarely hurts, rarely helps much past 200
4. **Deep, unpruned trees are ideal base estimators** — maximum variance to reduce
5. **Low-variance models (LR, shallow trees) gain little from bagging**
6. **Random Patches** offer maximum diversity — useful for high-dimensional data
7. **Pasting** (no replacement) works when training set is very large
8. **OOB score ≈ leave-one-out CV** — a reliable, cost-free estimate
9. **Random Forest is optimized bagging** — use it instead of manual BaggingClassifier for trees
10. **max_samples and max_features together** (Random Patches) control the diversity-accuracy tradeoff

---

## 🔗 Next Steps

- ➡️ `08_Ensemble_Learning/random_forest.ipynb` — Optimized bagging for trees
- ➡️ `08_Ensemble_Learning/boosting.ipynb` — Sequential ensemble (bias reduction)
- ➡️ `08_Ensemble_Learning/stacking.ipynb` — Meta-learning ensemble
- ➡️ `05_Model_Evaluation/bias_variance_tradeoff.md` — Theory behind variance reduction
